# The Degree Dividend: Early-Career ROI Across College Majors

## Overview
This notebook evaluates early-career economic returns (1 to 4 years post-graduation) across the top 50 most conferred Bachelor's degree fields in the United States.

### Data Sources
1. **IPEDS Completions (2022)**: Conferral counts by institution, award level, and 4-digit CIP code.
2. **U.S. Department of Education College Scorecard**: Field-of-study debt and 1-year / 4-year post-completion earnings.

### Methodology
- Standardize program taxonomy to 4-digit CIP codes (`XX.XX`).
- Restrict IPEDS conferrals to Bachelor's completions (`AWLEVEL = '05'`) and primary majors (`MAJORNUM = 1`).
- Aggregate Scorecard program-level cohorts using median debt and earnings, handling privacy suppression (`PS`) and computing 25th percentile downside risks.
- Derive economic indicators: Debt-to-Earnings ratios, benchmark earnings premiums, and early wage trajectory.

In [ ]:
# Core data manipulation and out-of-core SQL engine
import os
import io
import zipfile
import requests
import duckdb
import pandas as pd
import numpy as np

DATA_DIR = "raw_data"
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Workspace initialized. Working directory: '{DATA_DIR}'")

## 1. Data Ingestion
Download primary completions records from IPEDS. Scorecard field-of-study cohorts are unzipped directly into the working directory.

In [ ]:
def download_and_extract(url: str, destination: str) -> None:
    """Download a compressed zip archive and extract contents into the target directory."""
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, stream=True)
    response.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        archive.extractall(destination)

# Fetch IPEDS completions dataset (2021-2022 provisional/revised)
ipeds_url = "https://nces.ed.gov/ipeds/datacenter/data/C2022_A.zip"
download_and_extract(ipeds_url, DATA_DIR)

# Unpack local College Scorecard archive
for file_name in os.listdir(DATA_DIR):
    if file_name.endswith(".zip") and "Field" in file_name:
        with zipfile.ZipFile(os.path.join(DATA_DIR, file_name), 'r') as archive:
            archive.extractall(DATA_DIR)

print("Extracted datasets available in raw_data:")
for f in os.listdir(DATA_DIR):
    if f.endswith(".csv"):
        print(f" - {f}")

## 2. Identify the Top 50 Conferred Majors (IPEDS)
Filter completions to primary Bachelor's degrees (`AWLEVEL = '05'`, `MAJORNUM = 1`), strip aggregate categories, and truncate CIP codes to 4 digits (`XX.XX`).

In [ ]:
ipeds_path = os.path.join(DATA_DIR, "c2022_a_rv.csv")

# Aggregate conferral volume by 4-digit CIP
top_50_query = f"""
    WITH completions_filtered AS (
        SELECT
            SUBSTR(TRIM(CIPCODE), 1, 5) AS cip4,
            CTOTALT AS completions
        FROM read_csv_auto('{ipeds_path}')
        WHERE AWLEVEL = '05'                   -- Bachelor's degree awards
          AND MAJORNUM = 1                     -- Primary major to avoid duplicate headcounts
          AND TRIM(CIPCODE) NOT LIKE '99%'     -- Exclude summary/placeholder codes
    )
    SELECT
        cip4,
        SUM(completions) AS total_conferred
    FROM completions_filtered
    WHERE LENGTH(cip4) = 5
    GROUP BY cip4
    ORDER BY total_conferred DESC
    LIMIT 50
"""

top_50_df = duckdb.query(top_50_query).to_df()

# Map official CIP descriptions
title_mapping_query = """
    WITH clean_reference AS (
        SELECT
            TRIM(REPLACE(REPLACE(CIPCode, '=\"', ''), '\"', '')) AS clean_cip,
            TRIM(TRAILING '.' FROM TRIM(CIPTitle)) AS major_title
        FROM cip_titles_df
    )
    SELECT
        t.cip4,
        COALESCE(r.major_title, 'Title Unavailable') AS major_title,
        t.total_conferred
    FROM top_50_df t
    LEFT JOIN clean_reference r ON t.cip4 = r.clean_cip
    ORDER BY t.total_conferred DESC
"""

top_50_named = duckdb.query(title_mapping_query).to_df()
top_50_named.head()

## 3. Aggregate Financial Outcomes (College Scorecard)
Process program-level reporting for Bachelor's degrees (`CREDLEV = '3'`). Cast privacy suppression tokens (`PS`, `PrivacySuppressed`) to nulls and derive national median debt, 1-year/4-year earnings, and 25th percentile downside benchmarks.

In [ ]:
scorecard_path = os.path.join(DATA_DIR, "Most-Recent-Cohorts-Field-of-Study.csv")

scorecard_query = f"""
    WITH cleaned_scorecard AS (
        SELECT
            -- Normalize 4-digit CIP code from 'XXXX' to 'XX.XX'
            SUBSTR(LPAD(TRIM(CIPCODE), 4, '0'), 1, 2) || '.' || SUBSTR(LPAD(TRIM(CIPCODE), 4, '0'), 3, 2) AS cip4,
            TRY_CAST(
                CASE WHEN DEBT_ALL_STGP_EVAL_MDN IN ('PS', 'PrivacySuppressed', 'NULL') THEN NULL
                     ELSE DEBT_ALL_STGP_EVAL_MDN END AS DOUBLE
            ) AS debt_mdn,
            TRY_CAST(
                CASE WHEN EARN_MDN_1YR IN ('PS', 'PrivacySuppressed', 'NULL') THEN NULL
                     ELSE EARN_MDN_1YR END AS DOUBLE
            ) AS earn_1yr,
            TRY_CAST(
                CASE WHEN EARN_MDN_4YR IN ('PS', 'PrivacySuppressed', 'NULL') THEN NULL
                     ELSE EARN_MDN_4YR END AS DOUBLE
            ) AS earn_4yr
        FROM read_csv_auto('{scorecard_path}', all_varchar=True)
        WHERE CREDLEV = '3' -- Bachelor's degree cohort level
    )
    SELECT
        cip4,
        ROUND(MEDIAN(debt_mdn), 0) AS national_median_debt,
        ROUND(MEDIAN(earn_1yr), 0) AS national_median_earn_1yr,
        ROUND(MEDIAN(earn_4yr), 0) AS national_median_earn_4yr,
        ROUND(QUANTILE_CONT(earn_4yr, 0.25), 0) AS downside_risk_earn_4yr,
        COUNT(earn_4yr) AS reporting_programs_count
    FROM cleaned_scorecard
    GROUP BY cip4
    HAVING COUNT(earn_4yr) > 0
"""

scorecard_nat_df = duckdb.query(scorecard_query).to_df()
scorecard_nat_df.head()

## 4. Join and Feature Engineering
Combine volume and outcome metrics to compute relative financial KPIs:
- **Debt-to-Earnings Ratio**: Median Debt / Median 4-Year Earnings
- **Earnings Premium**: Percentage difference relative to the national all-bachelor's median
- **Early Wage Trajectory**: Growth from Year 1 to Year 4 earnings

In [ ]:
final_merge_query = """
    WITH merged AS (
        SELECT
            t.cip4,
            t.major_title,
            t.total_conferred,
            s.national_median_debt,
            s.national_median_earn_1yr,
            s.national_median_earn_4yr,
            s.downside_risk_earn_4yr,
            s.reporting_programs_count
        FROM top_50_named t
        LEFT JOIN scorecard_nat_df s ON t.cip4 = s.cip4
    ),
    benchmark AS (
        SELECT MEDIAN(national_median_earn_4yr) AS all_bachelors_median_4yr
        FROM scorecard_nat_df
    )
    SELECT
        m.cip4,
        m.major_title,
        m.total_conferred,
        m.national_median_debt,
        m.national_median_earn_1yr,
        m.national_median_earn_4yr,
        m.downside_risk_earn_4yr,
        ROUND(m.national_median_debt / NULLIF(m.national_median_earn_4yr, 0), 3) AS debt_to_earnings_ratio,
        ROUND(((m.national_median_earn_4yr - b.all_bachelors_median_4yr) / b.all_bachelors_median_4yr) * 100, 1) AS earnings_premium_pct,
        ROUND(((m.national_median_earn_4yr - m.national_median_earn_1yr) / NULLIF(m.national_median_earn_1yr, 0)) * 100, 1) AS early_wage_growth_pct,
        m.reporting_programs_count
    FROM merged m
    CROSS JOIN benchmark b
    ORDER BY m.total_conferred DESC
"""

final_df = duckdb.query(final_merge_query).to_df()
final_df.head(10)

## 5. Export Summary Dataset
Export the consolidated 50-row metrics dataset to feed the Streamlit dashboard.

In [ ]:
output_path = "top_50_majors_outcomes.csv"
final_df.to_csv(output_path, index=False)
print(f"Summary dataset exported to '{output_path}' ({os.path.getsize(output_path) / 1024:.1f} KB)")